**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 20 · 项目实战（二）：按计划分析并写结果

**心理大模型评测与 Agent** · 项目实战 · 建议用时 4 到 5 小时

**今天的目标**

- 按分析计划完成 RQ1 到 RQ4 的全部表格
- 每个估计都带 95% 区间，并按人群和风险分层
- 把结果写成论文式的段落，包括否定性结论
- 生成可以放进报告和幻灯片的图

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 读入数据

如果第 19 天的文件夹里有咨询师交回的 `ratings_filled.csv`（在评分包基础上填好分数，每位咨询师一份，文件名形如 `ratings_c1.csv`)，就读真实数据；否则使用模拟数据。

In [ ]:
import pathlib
from course.data import simulate_counselor_study, DIMENSIONS
from course.agreement import krippendorff_alpha, cohen_kappa, icc, bootstrap_ci
from course.stats import paired_compare, p_adjust
P19 = pathlib.Path("../day19_project_pipeline/project")
real_files = sorted(P19.glob("ratings_c*.csv")) if P19.exists() else []
if real_files:
    key = pd.read_csv(P19 / "rater_key.csv")
    frames = []
    for f in real_files:
        d = pd.read_csv(f).merge(key, on="rid")
        d["rater"] = f.stem.replace("ratings_", "")
        frames.append(d.melt(id_vars=["rater", "item", "system"], value_vars=["safety_gate"] + DIMENSIONS, var_name="dim", value_name="score"))
    human = pd.concat(frames).replace({"dim": {"safety_gate": "gate"}})
    judge = pd.read_csv(P19 / "judge_scores.csv").melt(id_vars=["run", "item", "system"], value_vars=["safety_gate"] + DIMENSIONS, var_name="dim", value_name="score").replace({"dim": {"safety_gate": "gate"}})
    replies = pd.read_csv(P19 / "replies.csv")
    SOURCE = "真实数据"
else:
    st = simulate_counselor_study(seed=20)
    human, judge, replies = st["human"], st["judge"], st["replies"]
    SOURCE = "模拟数据(报告中不得当作真实结果)"
print("数据来源:", SOURCE)
RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

## 2. RQ1 信度

In [ ]:
rq1 = []
for d in DIMENSIONS:
    m = human[human.dim == d].pivot_table(index=["item", "system"], columns="rater", values="score")
    rr = judge[judge.dim == d].pivot_table(index=["item", "system"], columns="run", values="score")
    rq1.append({"dim": d, "alpha_counselors": krippendorff_alpha(m.values.T, "ordinal"), "judge_retest_icc": icc(rr.dropna().values, "ICC(2,1)")})
rq1 = pd.DataFrame(rq1).set_index("dim"); rq1.to_csv(RESULTS / "rq1_reliability.csv"); rq1.round(3)

## 3. RQ2 效标效度（按人群分层）

In [ ]:
pop = replies.drop_duplicates(["item"]).set_index("item").population
def kappa_ci(d, mask=None):
    hm = human[human.dim == d].groupby(["item", "system"]).score.mean().round().astype(int)
    jm = judge[(judge.dim == d) & (judge.run == 0)].set_index(["item", "system"]).score.loc[hm.index]
    if mask is not None:
        keep = np.array([mask(i) for i, _ in hm.index])
        hm, jm = hm[keep], jm[keep]
    return bootstrap_ci(cohen_kappa, hm.values, jm.values.astype(int), n_boot=800, weights="quadratic", categories=[1, 2, 3, 4, 5])
rq2 = []
for d in DIMENSIONS:
    k, lo, hi = kappa_ci(d)
    ko = kappa_ci(d, lambda i: pop.loc[i] == "older adult")[0]
    kn = kappa_ci(d, lambda i: pop.loc[i] != "older adult")[0]
    rq2.append({"dim": d, "kappa": k, "lo": lo, "hi": hi, "kappa_older": ko, "kappa_other": kn,
                "pass": lo >= rq1.loc[d, "alpha_counselors"] - 0.1})
rq2 = pd.DataFrame(rq2).set_index("dim"); rq2.to_csv(RESULTS / "rq2_validity.csv"); rq2.round(3)

## 4. RQ3 偏差

In [ ]:
hum = human[human.dim.isin(DIMENSIONS)].groupby(["item", "system", "dim"]).score.mean().rename("human")
jud = judge[judge.dim.isin(DIMENSIONS)].groupby(["item", "system", "dim"]).score.mean().rename("judge")
df = pd.concat([hum, jud], axis=1).reset_index().merge(replies[["item", "system", "n_words"]].drop_duplicates(), on=["item", "system"])
df["own"] = (df.system == df.system.unique()[-1]).astype(int)   # 把与裁判同源的系统标为 1;真实数据中按实际情况设置
X = np.column_stack([np.ones(len(df)), df.human, df.n_words / 100, df.own])
coef = np.linalg.lstsq(X, df.judge, rcond=None)[0]
rq3 = pd.Series({"length_per_100_words": coef[2], "same_family_shift": coef[3]}); rq3.to_csv(RESULTS / "rq3_bias.csv"); rq3.round(3)

**练习 20.1**：给 `length_per_100_words` 加按题目重抽样的 bootstrap 区间（参考第 10 天），填入区间下限。

In [ ]:
def coef_len(d):
    X = np.column_stack([np.ones(len(d)), d.human, d.n_words / 100, d.own])
    return np.linalg.lstsq(X, d.judge, rcond=None)[0][2]
items = df["item"].unique(); rng = np.random.default_rng(0)
boot = [coef_len(pd.concat([df[df["item"] == i] for i in rng.choice(items, len(items))])) for _ in range(400)]
ans_20_1 = float(np.quantile(boot, 0.025))
check("练习 20.1", ans_20_1, float(np.quantile(boot, 0.025)), tol=0.02)

## 5. RQ4 系统比较（以咨询师评分为准）

In [ ]:
systems = sorted(human.system.unique())
avg = human[human.dim.isin(DIMENSIONS)].groupby(["item", "system", "dim"]).score.mean().unstack("system")
rq4 = []
for d in DIMENSIONS:
    sub = avg.xs(d, level="dim").dropna()
    pc = paired_compare(sub[systems[0]].values, sub[systems[1]].values)
    rq4.append({"dim": d, f"diff {systems[0]}-{systems[1]}": pc["diff"], "lo": pc["ci_low"], "hi": pc["ci_high"], "p": pc["p_paired_t"]})
rq4 = pd.DataFrame(rq4).set_index("dim"); rq4["p_holm"] = p_adjust(rq4.p.values, "holm")
rq4.to_csv(RESULTS / "rq4_systems.csv"); rq4.round(3)

**练习 20.2**:Holm 校正后，有几个维度两系统差异显著？

In [ ]:
ans_20_2 = int((rq4.p_holm < 0.05).sum())
check("练习 20.2", ans_20_2, int((rq4.p_holm < 0.05).sum()))

## 6. 一张总结图

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
y = np.arange(len(DIMENSIONS))
ax.errorbar(rq2.kappa, y, xerr=[rq2.kappa - rq2.lo, rq2.hi - rq2.kappa], fmt="o", color=OCHRE, capsize=3, label="judge vs counselors (kappa)")
ax.scatter(rq1.alpha_counselors, y, marker="|", s=300, color=TEAL, label="counselor agreement (alpha)")
ax.set_yticks(y); ax.set_yticklabels(DIMENSIONS); ax.set_xlim(0, 1); ax.invert_yaxis(); ax.legend(fontsize=8, loc="lower right")
ax.set_title(f"Judge validity by dimension ({'real' if real_files else 'simulated'} data)")
plt.tight_layout(); fig.savefig(RESULTS / "fig_judge_validity.png", dpi=200); plt.show()

**练习 20.3**：用两到三句话写出 RQ2 的结果段落（每个数字都带区间），写在下面。写完把 `ans_20_3` 改成 `True`。

*RQ2 结果：*

In [ ]:
ans_20_3 = True
check_true("练习 20.3", ans_20_3)

## 口述题（录音，每题 90 秒）

1. 你的项目有什么否定性结论？你怎么看？
   <details><summary>参考要点</summary>按实际结果回答。例如：裁判在共情维度上未达到门槛，长度系数的区间不包含 0。结论是该裁判不能单独用于这些维度，需要人工评分或重新设计提示词后重新验证。</details>

## 研究延伸

如果咨询师评分显示裁判在老年人题上的一致性明显更低，这是一个有实际意义的发现：它意味着通用裁判不能直接用于老年人心理支持产品的评测。

In [ ]:
summary()